# Network Analysis (Hardy Cross) & Water Hammer

Two advanced hydraulic analyses beyond the single-pipe case:

1. **Hardy Cross method** — flow distribution in closed-loop pipe networks (multiple interconnected pipes, not just one line)
2. **Water hammer (Joukowsky equation)** — transient pressure surges from sudden valve closure or pump trip, a genuine pipe-rupture safety check

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

## Part 1: Hardy Cross Network Analysis

### Validation: the classic 4-node, 2-loop textbook example

Before trusting the solver on a real network, reproduce a published worked example exactly (Wikipedia: Hardy Cross method) — a network with an abstract head-loss law just to verify the loop-balancing algorithm itself.

In [ ]:
from src.hydraulics.network import hardy_cross_solve, Loop, LoopMember

r = {'12': 1, '13': 5, '23': 1, '24': 5, '34': 1}
def head_loss_fn(name, Q):
    return r[name] * Q**2

initial_flows = {'12': 5.0, '13': 5.0, '23': 0.0, '24': 5.0, '34': 5.0}
loop123 = Loop('123', [LoopMember('12', +1), LoopMember('23', +1), LoopMember('13', -1)])
loop234 = Loop('234', [LoopMember('23', -1), LoopMember('24', +1), LoopMember('34', -1)])

result = hardy_cross_solve(
    pipe_names=['12','13','23','24','34'], loops=[loop123, loop234],
    head_loss_fn=head_loss_fn, initial_flows=initial_flows, n=2.0,
)
print(f'Converged in {result.iterations} iteration(s)')
for name, Q in result.flows.items():
    print(f'  Q{name} = {Q:.4f} L/s')

Matches the published solution (6.6667, 3.3333, 3.3333, 3.3333, 6.6667 L/s) exactly, including converging in essentially one iteration — confirming the solver's loop-update convention matches the standard method.

### A more realistic network

Same topology (4 nodes, 5 pipes, 2 loops), but now with real pipe diameters/lengths/roughness, solved with actual Darcy-Weisbach head losses via `PipeNetwork`.

In [ ]:
from src.hydraulics.network import PipeNetwork, NetworkPipe
from src.utils.constants import WATER_DENSITY, WATER_VISCOSITY, PVC_ROUGHNESS

pipes = [
    NetworkPipe('12', '1', '2', diameter_m=0.10, length_m=200.0, roughness_m=PVC_ROUGHNESS),
    NetworkPipe('13', '1', '3', diameter_m=0.075, length_m=250.0, roughness_m=PVC_ROUGHNESS),
    NetworkPipe('23', '2', '3', diameter_m=0.05, length_m=150.0, roughness_m=PVC_ROUGHNESS),
    NetworkPipe('24', '2', '4', diameter_m=0.075, length_m=200.0, roughness_m=PVC_ROUGHNESS),
    NetworkPipe('34', '3', '4', diameter_m=0.10, length_m=200.0, roughness_m=PVC_ROUGHNESS),
]
loop123 = Loop('123', [LoopMember('12', +1), LoopMember('23', +1), LoopMember('13', -1)])
loop234 = Loop('234', [LoopMember('23', -1), LoopMember('24', +1), LoopMember('34', -1)])

network = PipeNetwork(pipes, [loop123, loop234], density=WATER_DENSITY, viscosity=WATER_VISCOSITY)

**Poka-Yoke**: always verify the initial flow guess satisfies node continuity before solving — a bad guess here would silently converge to a wrong (or non-physical) answer.

In [ ]:
initial_flows = {'12': 0.030, '13': 0.020, '23': 0.0, '24': 0.030, '34': 0.020}
external = {'1': 0.050, '4': -0.050}   # 50 L/s supplied at node 1, drawn off at node 4

residuals = network.check_node_continuity(initial_flows, external)
print('Continuity residuals (should all be ~0):')
for node, r in residuals.items():
    print(f'  Node {node}: {r:.6f} m3/s')

In [ ]:
result = network.solve(initial_flows)
print(f'Converged: {result.converged} ({result.iterations} iterations)')
print()
df = pd.DataFrame({
    'pipe': list(result.flows.keys()),
    'flow_Ls': [v*1000 for v in result.flows.values()],
    'head_loss_m': list(result.head_losses.values()),
})
df

Note how flow naturally distributes more through the larger-diameter, lower-resistance path (12-24) versus the narrower 13-34 path, and the shared cross-connection pipe 23 carries comparatively little flow — exactly the kind of insight Hardy Cross was built to provide without needing to solve the full nonlinear system of equations directly.

## Part 2: Water Hammer Analysis

What pressure surge results from a valve closing on the existing 4-inch baseline scenario, and how much does pipe material and closure speed matter?

In [ ]:
from src.hydraulics.transients import evaluate_water_hammer
from src.utils.constants import WATER_BULK_MODULUS_PA, STEEL_YOUNGS_MODULUS_PA, PVC_YOUNGS_MODULUS_PA
from src.utils.validation import check_water_hammer_risk

# A 4-inch pipe, 200 m to the nearest reservoir, water hammer from
# a valve that brings flow from 2 m/s to a stop.
common = dict(
    bulk_modulus_Pa=WATER_BULK_MODULUS_PA, density=WATER_DENSITY,
    diameter_m=0.1016, wall_thickness_m=0.006, length_m=200.0,
    delta_v_m_s=2.0, initial_pressure_Pa=300_000,
)

for material, E in [('PVC', PVC_YOUNGS_MODULUS_PA), ('Steel', STEEL_YOUNGS_MODULUS_PA)]:
    for closure_s, label in [(0.05, 'fast (0.05s)'), (3.0, 'slow (3.0s)')]:
        r = evaluate_water_hammer(**common, youngs_modulus_Pa=E, closure_time_s=closure_s)
        print(f'{material:6s} | {label:14s} | wave speed={r.wave_speed_m_s:7.1f} m/s | '
              f'rapid={r.is_rapid_closure!s:5s} | surge={r.surge_Pa/1000:8.1f} kPa | '
              f'peak={r.peak_pressure_Pa/1000:8.1f} kPa')

## The key safety insight

Two levers dramatically change the outcome of the same physical event (a 2 m/s velocity change):

1. **Pipe material**, *for fast closures*: steel's higher stiffness gives a faster wave speed, and a correspondingly higher surge (~2720 kPa) than the more flexible PVC pipe (~845 kPa) for the same fast closure — PVC's elasticity absorbs some of the shock.
2. **Closure speed**: slowing the closure from 0.05s to 3.0s (well beyond the pipe's critical period) cuts the steel-pipe surge by roughly 90% and the PVC-pipe surge by roughly 70%.

More precisely: in the *slow*-closure regime, the wave speed `a` cancels out of the Michaud approximation entirely (`surge = rho * dv * 2L / closure_time`), so both materials converge to the **same** ~266 kPa surge regardless of pipe stiffness — visible directly in the table above. Pipe material only matters for fast (near-instantaneous) closures; for slow ones, closure time and pipe length are what matter.

In [ ]:
# Check the fast-closure steel case against a typical PN10 (1000 kPa) pipe rating
r_fast_steel = evaluate_water_hammer(**common, youngs_modulus_Pa=STEEL_YOUNGS_MODULUS_PA,
                                       closure_time_s=0.05)
warning = check_water_hammer_risk(r_fast_steel.peak_pressure_Pa, pipe_rated_pressure_Pa=1_000_000)
print(f'Peak pressure: {r_fast_steel.peak_pressure_Pa/1000:.1f} kPa')
print(warning or 'Within safe operating margin.')

## Caveats

- The Hardy Cross solver requires an initial flow guess that already satisfies node continuity — always check with `check_node_continuity` first.
- Water hammer here uses the Joukowsky equation with a simplified linear-closure approximation for slow closures. Real transient analysis (method of characteristics) accounts for wave reflections, friction, and non-linear valve closure curves — use this module for first-pass screening, not final design certification on critical systems.